# 엔티티 추출 비교: Hybrid와 LLM-only

본 실험은 [01. Short-Term Memory](../notebooks/01_short_term_memory.ipynb)의 한국어 메시지 4개에 대해 현재 추출 설정(GLiNER + LLM 파이프라인, 이하 Hybrid)과 LLM 단독 추출의 결과를 비교한다.
노트북 01을 실행했을 때 `GCP에서`, `오라클(PERSON)`과 같은 엔티티가 저장되는 원인이 어느 단계에 있는지 확인하는 것이 목적이다.

Neo4j에는 연결하지 않는다. 저장 직전 단계의 필터(`filter_invalid_entities`, `_apply_ontology`)와 엔티티 해소를 빈 그래프에 대해 재현하여, 실제로 저장될 결과를 DB 변경 없이 관찰한다.

## 결과 요약

> 아래 수치는 2026-11-09 실행 결과(`entity_extraction_comparison.json`, 메시지당 3회 반복)이다.

| 관찰 항목 | LLM-only | Hybrid | Hybrid 내부 LLM 단계 |
|---|---|---|---|
| `GCP`(ORGANIZATION) | 3/3 | 3/3 | 3/3 |
| `Cloud DNS`(OBJECT) | 3/3 | 3/3 | 3/3 |
| `오라클`(ORGANIZATION) | 3/3 | 3/3 | 3/3 |
| **`GCP에서`** (조사 포함) | 0/3 | **3/3** | 0/3 |
| **`오라클`(PERSON)** | 0/3 | **3/3** | 0/3 |
| **`IP 기능을`** (조사 포함) | 0/3 | **3/3** | 0/3 |
| 관계 출발점이 `오라클(PERSON)`으로 선택됨 | 0/3 | 1/3 | – |

- 잘못된 엔티티 3종은 **모두 GLiNER 단계에서 생성된다**. 같은 실행의 LLM 단계 응답에는 한 번도 포함되지 않았다.
- CONFIDENCE 병합의 중복 판정 키는 `정규화된 이름::타입`이다. 따라서 `GCP에서`와 `GCP`, `오라클(PERSON)`과 `오라클(ORGANIZATION)`은 서로 다른 엔티티로 함께 남는다.
- GLiNER가 부여한 confidence는 `GCP에서` 0.87, `오라클(PERSON)` 0.86으로 높다. `confidence_threshold`를 올려도 `IP 기능을`(0.50)만 제거되며 나머지 두 오류는 남는다.
- Hybrid가 LLM-only보다 정답을 더 찾은 사례는 없다.

## 실험 조건

| 항목 | 값 |
|---|---|
| SDK | `neo4j-agent-memory` 0.8.0 |
| LLM | `gpt-5.4-mini` |
| Hybrid 구성 | GLiNER `fastino/gliner2.5-base-v1`(threshold 0.2) → LLM, `merge_strategy=confidence` |
| 온톨로지 | POLE+O (`PERSON`, `LOCATION`, `ORGANIZATION`, `EVENT`, `OBJECT`) |
| `confidence_threshold` | 0.0 |
| 반복 | 메시지 4개 × 3회 × 2개 모드, 반복마다 모드 실행 순서를 교대 |
| DB 연결 | 없음 (`EmptyGraph`) |

두 모드는 같은 모델·온톨로지·프롬프트·저장 필터를 사용하며 LLM 호출은 서로 독립이다. LLM 응답의 무작위성을 배제하기 위해 Hybrid 파이프라인의 LLM 단계 결과(`paired_llm_stage`)를 함께 기록하여, 동일한 LLM 응답에 GLiNER 결과만 더해진 효과를 따로 비교한다.

| # | 입력 메시지 |
|---|---|
| 1 | 이제 GCP에서 매달 나가는 비용은 Cloud DNS 영역 3개, 월 약 $0.6뿐입니다. |
| 2 | 화면과 상관없는 서버 쪽 작업(공지 저장, 작업실 정보)은 이미 같이 진행하고 있어요. |
| 3 | 11월 사용량을 서비스별로 조회해 보니 금액이 붙은 항목이 하나도 없습니다. |
| 4 | 오라클 공식 입장: 고정 IP 기능을 처음 발표할 때 'There is no charge to use this feature'라고 밝혔습니다. |

## 준비

### 로그 출력 설정

Neo4j 서버 알림과 Transformers 모델 설정 모듈의 로그를 `ERROR` 이상으로 제한한다.

In [ ]:
import logging

logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)
logging.getLogger("transformers.configuration_utils").setLevel(logging.ERROR)

### 프로젝트 경로와 메모리 설정

현재 작업 디렉터리에서 상위로 올라가며 `pyproject.toml`이 있는 프로젝트 루트를 찾아 모듈 검색 경로에 추가한다.
`load_memory_settings()`는 노트북 01과 동일한 추출 설정을 구성한다.

In [ ]:
from pathlib import Path
import sys

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv

from config.memory_client import load_memory_settings

load_dotenv(PROJECT / ".env")
settings = load_memory_settings()
RESULT_PATH = PROJECT / "experiments/entity_extraction_comparison.json"
print(f"✓ Project root: {PROJECT}")
print(f"✓ Result file: {RESULT_PATH.relative_to(PROJECT)}")

### 실행 방식 선택

`RUN_EXPERIMENT = False`이면 저장된 결과 파일을 불러와 분석만 수행한다.
`True`로 바꾸면 LLM과 임베딩 API를 호출하여 실험을 다시 실행하고 결과 파일을 덮어쓴다. 메시지 4개 × `REPEATS`회 × 2개 모드만큼 LLM이 호출되며, GLiNER 모델 로딩이 포함된다.

In [ ]:
RUN_EXPERIMENT = False
REPEATS = 3

### 보조 객체

- `EmptyGraph`: 모든 조회에 빈 결과를 반환하고 쓰기를 거부한다. 엔티티 해소가 기존 노드 없이 메시지 내부에서만 동작하도록 한다.
- `notebook_messages()`: 노트북 01의 `messages` 리터럴을 그대로 읽어 입력 문장을 동기화한다.
- `save()`: 실행 중간 결과를 포함하여 보고서를 JSON으로 저장한다.

In [ ]:
import ast
import json


class EmptyGraph:
    """No stored candidates; resolution runs only within each input message."""

    async def execute_read(self, query, parameters=None):
        return []

    async def execute_write(self, *args, **kwargs):
        raise RuntimeError("Database writes are disabled in this experiment")


def notebook_messages():
    """notebooks/01_short_term_memory.ipynb의 `messages` 리터럴을 읽는다."""
    notebook = json.loads((PROJECT / "notebooks/01_short_term_memory.ipynb").read_text())
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        tree = ast.parse("".join(cell["source"]))
        for node in tree.body:
            if isinstance(node, ast.Assign) and any(
                isinstance(target, ast.Name) and target.id == "messages"
                for target in node.targets
            ):
                return ast.literal_eval(node.value)
    raise ValueError("No literal messages list found in notebook 01")


def save(path, report):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")

## 실험

### 추출기와 저장 필터 구성

Hybrid는 현재 설정 그대로, LLM-only는 `extractor_type`만 `LLM`으로 바꾼 설정으로 추출기를 만든다.
`prepare()`는 SDK의 `ShortTermMemory`가 저장 직전에 적용하는 유효성 필터와 온톨로지 정규화를 재현한다. 이 함수는 SDK 내부 메서드를 사용하므로, 결과 파일에 기록한 SDK 버전에서만 동작이 보장된다.

In [ ]:
from neo4j_agent_memory import ExtractorType
from neo4j_agent_memory.extraction.factory import create_extractor, resolve_ontology
from neo4j_agent_memory.extraction.pipeline import ExtractionPipeline
from neo4j_agent_memory.memory.short_term import ShortTermMemory


def build_components():
    ontology = resolve_ontology(settings.extraction)
    configs = {
        "hybrid": settings.extraction,
        "llm_only": settings.extraction.model_copy(update={"extractor_type": ExtractorType.LLM}),
    }
    extractors = {
        name: create_extractor(config, settings.schema_config, settings.llm, ontology=ontology)
        for name, config in configs.items()
    }
    assert isinstance(extractors["hybrid"], ExtractionPipeline)
    storage = ShortTermMemory(
        EmptyGraph(), ontology=ontology,
        validation_mode=settings.schema_config.validation_mode,
    )

    def prepare(result):
        return storage._apply_ontology(result.filter_invalid_entities(ontology))

    return ontology, configs, extractors, storage, prepare

### 반복 추출

각 메시지를 두 모드로 추출하고, 반복마다 모드 실행 순서를 바꾼다.
Hybrid는 `extract_with_details()`로 GLiNER 단계와 LLM 단계의 결과를 각각 기록하며, 단계 실패나 단계 수 불일치가 있으면 중단한다.
매 실행 후 보고서를 저장하므로 중간에 실패해도 그때까지의 결과가 남는다.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter


async def run_extraction(repeats, configs, extractors, prepare):
    report = {
        "started_at": datetime.now(timezone.utc).isoformat(),
        "library_version": version("neo4j-agent-memory"),
        "notebook": "notebooks/01_short_term_memory.ipynb",
        "llm_model": settings.extraction.llm_model,
        "repeats": repeats,
        "database_connected": False,
        "configs": {name: config.model_dump(mode="json") for name, config in configs.items()},
        "resolution_config": settings.resolution.model_dump(mode="json"),
        "messages": notebook_messages(),
        "runs": [],
        "status": "running",
    }
    save(RESULT_PATH, report)
    total = repeats * len(report["messages"]) * len(extractors)
    for repeat in range(1, repeats + 1):
        for index, message in enumerate(report["messages"], start=1):
            # Alternate which mode runs first across repetitions.
            modes = list(extractors) if repeat % 2 else list(reversed(extractors))
            for mode in modes:
                extractor = extractors[mode]
                started = perf_counter()
                entry = {"repeat": repeat, "message_index": index, "mode": mode}
                try:
                    if isinstance(extractor, ExtractionPipeline):
                        details = await extractor.extract_with_details(message)
                        entry["stages"] = [
                            {
                                "name": stage.stage_name, "success": stage.success,
                                "error": stage.error, "duration_ms": stage.duration_ms,
                                "result": stage.result.model_dump(mode="json"),
                                "prepared_result": prepare(stage.result).model_dump(mode="json"),
                            }
                            for stage in details.stage_results
                        ]
                        if not all(stage.success for stage in details.stage_results):
                            raise RuntimeError("An extraction stage failed; inspect stages")
                        if len(details.stage_results) != 2:
                            raise RuntimeError("Expected GLiNER and LLM stages")
                        raw = details.final_result
                    else:
                        raw = await extractor.extract(message)
                    prepared = prepare(raw)
                    entry.update(
                        raw_result=raw.model_dump(mode="json"),
                        prepared_result=prepared.model_dump(mode="json"),
                        elapsed_seconds=round(perf_counter() - started, 3),
                    )
                except Exception as exc:
                    entry["error"] = f"{type(exc).__name__}: {exc}"
                    report["runs"].append(entry)
                    report["status"] = "failed"
                    save(RESULT_PATH, report)
                    raise
                report["runs"].append(entry)
                save(RESULT_PATH, report)
                entities = ", ".join(f"{e.name}({e.type})" for e in prepared.entities) or "(none)"
                print(f"{len(report['runs'])}/{total} {mode} message={index} "
                      f"{entry['elapsed_seconds']}s: {entities}")
    return report

### 엔티티 해소와 관계 endpoint 추적

추출 결과가 그래프에 어떻게 들어갈지 저장 없이 추적한다.

- **엔티티 해소**: 등장한 이름마다 임베딩을 한 번만 계산하여 재사용하고, `OntologyResolver.resolve_episode()`를 빈 그래프에 대해 메시지 단위로 실행한다. 같은 메시지 안의 `GCP`와 `GCP에서`가 병합되는지 확인할 수 있다.
- **관계 endpoint**: 관계의 출발점·도착점 이름이 같은 이름의 여러 노드(예: `오라클` PERSON/ORGANIZATION) 중 어느 노드로 연결되는지를 SDK의 `_local_endpoints()`로 재현한다. 병합 이후의 ID 재매핑은 재현하지 않는다.

In [ ]:
from dataclasses import asdict

from neo4j_agent_memory.extraction.base import ExtractionResult
from neo4j_agent_memory.resolution.ontology import OntologyResolver


async def trace_resolution(report, ontology, storage):
    names = sorted({
        entity["name"] for entry in report["runs"]
        for entity in entry["prepared_result"]["entities"]
    })
    vectors = await settings.embedding.embed(names) if names else []
    embeddings = dict(zip(names, vectors, strict=True))
    resolver = OntologyResolver(EmptyGraph(), ontology=ontology, config=settings.resolution)
    for entry in report["runs"]:
        result = ExtractionResult.model_validate(entry["prepared_result"])
        decisions = await resolver.resolve_episode(
            result.entities, embeddings=[embeddings[entity.name] for entity in result.entities]
        )
        entry["empty_graph_per_message_resolution"] = [
            {"name": entity.name, "type": entity.type, **asdict(decision)}
            for entity, decision in zip(result.entities, decisions, strict=True)
        ]

        # Trace the SDK's local endpoint choice without persisting any node.
        name_to_id, mention_to_id, typed_nodes, by_id = {}, {}, {}, {}
        for index, entity in enumerate(result.entities):
            node_id = f"diagnostic:{index}"
            by_id[node_id] = {"name": entity.name, "type": entity.type}
            key = entity.name.lower().strip()
            name_to_id[key] = node_id
            typed_nodes.setdefault(key, []).append((entity.type.upper(), entity.subtype, node_id))
            if entity.id is not None:
                mention_to_id[entity.id] = node_id
        local_edges = []
        for relation in result.relations:
            source = mention_to_id.get(relation.source_id)
            target = mention_to_id.get(relation.target_id)
            if source is None or target is None:
                source, target = storage._local_endpoints(
                    relation.source.lower().strip(), relation.relation_type,
                    relation.target.lower().strip(), name_to_id, typed_nodes,
                )
            local_edges.append({
                "relation_type": relation.relation_type,
                "source": by_id.get(source), "target": by_id.get(target),
                "note": "Local endpoint selection before persistence; no graph writes",
            })
        entry["local_relation_endpoints"] = local_edges

### 관찰 지표 집계

4개 메시지는 정답 코퍼스가 아니므로 정밀도·재현율을 계산하지 않는다.
대신 노트북 01에서 문제가 된 엔티티가 모드별로 몇 번의 실행에서 나타났는지를 센다.

In [ ]:
def summarize(report):
    """Count observed cases, without treating four messages as a gold corpus."""
    summary = {}
    for mode in ("hybrid", "llm_only", "paired_llm_stage"):
        selected = []
        for run in report["runs"]:
            if mode == "paired_llm_stage" and run["mode"] == "hybrid":
                stage = next(s for s in run["stages"] if s["name"] == "LLMEntityExtractor")
                selected.append({**run, "prepared_result": stage["prepared_result"]})
            elif run["mode"] == mode:
                selected.append(run)

        def count(message_index, name, entity_type=None):
            return sum(
                any(e["name"] == name and (entity_type is None or e["type"] == entity_type)
                    for e in run["prepared_result"]["entities"])
                for run in selected if run["message_index"] == message_index
            )

        summary[mode] = {
            "runs_per_message": report["repeats"],
            "gcp_with_particle": count(1, "GCP에서"),
            "gcp_organization": count(1, "GCP", "ORGANIZATION"),
            "cloud_dns_object": count(1, "Cloud DNS", "OBJECT"),
            "oracle_person": count(4, "오라클", "PERSON"),
            "oracle_organization": count(4, "오라클", "ORGANIZATION"),
            "ip_feature_with_particle": count(4, "IP 기능을"),
            "fixed_ip_feature": count(4, "고정 IP 기능"),
        }
        if mode != "paired_llm_stage":
            summary[mode]["oracle_person_local_endpoint"] = sum(
                any(edge["source"] == {"name": "오라클", "type": "PERSON"}
                    for edge in run["local_relation_endpoints"])
                for run in selected if run["message_index"] == 4
            )
    return summary

### 실행 또는 결과 불러오기

`RUN_EXPERIMENT`에 따라 실험 전체를 실행하거나 저장된 보고서를 불러온다.

In [ ]:
if RUN_EXPERIMENT:
    ontology, configs, extractors, storage, prepare = build_components()
    report = await run_extraction(REPEATS, configs, extractors, prepare)
    await trace_resolution(report, ontology, storage)
    report["summary"] = summarize(report)
    report["status"] = "complete"
    report["completed_at"] = datetime.now(timezone.utc).isoformat()
    report["notes"] = [
        "Both modes use the same model, ontology, prompt and storage filters; LLM calls are independent.",
        "Hybrid stages also provide a paired comparison using exactly the same LLM response.",
        "Resolution has no stored candidates and runs independently per message; this is not an end-to-end DB test.",
        "The first hybrid duration includes lazy GLiNER model loading; timings are not a latency benchmark.",
        "Local relation endpoint tracing precedes persistence and does not emulate ID remapping after a merge.",
    ]
    save(RESULT_PATH, report)
else:
    report = json.loads(RESULT_PATH.read_text())

print(f"✓ status={report['status']} runs={len(report['runs'])} "
      f"sdk={report['library_version']} llm={report['llm_model']}")

## 결과 분석

표는 `report`에서 다시 계산한다. 셀 출력 없이 읽을 수 있도록, 각 표 아래에 2026-11-09 실행의 값과 해석을 함께 기록한다.

In [ ]:
from collections import Counter

from IPython.display import Markdown, display


def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def entity_summary(runs, with_confidence=False):
    """(name, type)별 등장 실행 수와 최대 confidence를 `<br>`로 이어 붙인다."""
    counts, confidence = Counter(), {}
    for run in runs:
        for entity in run["prepared_result"]["entities"]:
            key = (entity["name"], entity["type"])
            confidence[key] = max(confidence.get(key, 0), entity["confidence"])
        counts.update({(e["name"], e["type"]) for e in run["prepared_result"]["entities"]})
    items = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return "<br>".join(
        f"{name}({entity_type}) {count}/{len(runs)}"
        + (f" · {confidence[name, entity_type]:.2f}" if with_confidence else "")
        for (name, entity_type), count in items
    ) or "–"

### 관찰 지표

모드별로 각 엔티티가 3회 중 몇 번의 실행에서 나타났는지 표시한다.

In [ ]:
labels = {
    "gcp_organization": "`GCP`(ORGANIZATION)",
    "gcp_with_particle": "`GCP에서` (조사 포함)",
    "cloud_dns_object": "`Cloud DNS`(OBJECT)",
    "oracle_organization": "`오라클`(ORGANIZATION)",
    "oracle_person": "`오라클`(PERSON)",
    "ip_feature_with_particle": "`IP 기능을` (조사 포함)",
    "fixed_ip_feature": "`고정 IP 기능`",
    "oracle_person_local_endpoint": "관계 출발점 = `오라클`(PERSON)",
}
modes = ["llm_only", "hybrid", "paired_llm_stage"]
summary = report["summary"]
show_table(
    ["지표", *modes],
    [[label, *(f"{summary[m][key]}/{summary[m]['runs_per_message']}" if key in summary[m] else "–"
               for m in modes)]
     for key, label in labels.items()],
)

**해석**

- LLM-only와 Hybrid 내부 LLM 단계(`paired_llm_stage`)는 모든 지표에서 같은 값을 보인다. 두 모드의 차이는 LLM 응답의 무작위성이 아니라 GLiNER 결과의 추가로 설명된다.
- `고정 IP 기능`은 세 모드 모두 1/3에 그친다. 이 엔티티의 누락은 LLM 응답의 변동이며, GLiNER는 이를 보완하지 못하고 조사가 붙은 `IP 기능을`을 대신 추가한다.

### 메시지별 저장 대상 엔티티

저장 필터를 적용한 최종 엔티티를 메시지·모드별로 비교한다. `n/3`은 3회 실행 중 등장한 횟수이다.

In [ ]:
show_table(
    ["#", "메시지", "LLM-only", "Hybrid"],
    [
        [index, message,
         *(entity_summary([r for r in report["runs"]
                           if r["mode"] == mode and r["message_index"] == index])
           for mode in ("llm_only", "hybrid"))]
        for index, message in enumerate(report["messages"], start=1)
    ],
)

| # | LLM-only | Hybrid |
|---|---|---|
| 1 | GCP(ORGANIZATION) 3/3<br>Cloud DNS(OBJECT) 3/3 | GCP(ORGANIZATION) 3/3<br>Cloud DNS(OBJECT) 3/3<br>**GCP에서(ORGANIZATION) 3/3** |
| 2 | 공지 저장(EVENT) 3/3<br>작업실 정보(OBJECT) 3/3<br>서버 쪽 작업(EVENT) 1/3 | 공지 저장(EVENT) 3/3<br>작업실 정보(OBJECT) 3/3<br>서버 쪽 작업(OBJECT) 1/3 |
| 3 | – | – |
| 4 | 오라클(ORGANIZATION) 3/3<br>고정 IP 기능(OBJECT) 1/3 | 오라클(ORGANIZATION) 3/3<br>**오라클(PERSON) 3/3**<br>**IP 기능을(OBJECT) 3/3**<br>고정 IP 기능(OBJECT) 1/3 |

**해석**

- 메시지 1과 4에서 Hybrid는 LLM-only의 결과를 모두 포함하고, 잘못된 엔티티를 추가로 가진다.
- 메시지 2의 `서버 쪽 작업` 타입 차이(EVENT/OBJECT)는 두 모드의 독립적인 LLM 호출에서 생긴 변동이다. GLiNER는 메시지 2와 3에서 아무 엔티티도 반환하지 않았다.

### GLiNER 단계의 기여

Hybrid 실행에서 GLiNER 단계가 단독으로 반환한 엔티티와 최대 confidence를 표시한다.

In [ ]:
def gliner_stage(run):
    stage = next(s for s in run["stages"] if s["name"] != "LLMEntityExtractor")
    return {"prepared_result": stage["prepared_result"]}


show_table(
    ["#", "GLiNER 단계 엔티티 · confidence"],
    [[index, entity_summary([gliner_stage(r) for r in report["runs"]
                             if r["mode"] == "hybrid" and r["message_index"] == index],
                            with_confidence=True)]
     for index in range(1, len(report["messages"]) + 1)],
)
print("gliner_threshold:", report["configs"]["hybrid"]["gliner_threshold"],
      "/ confidence_threshold:", report["configs"]["hybrid"]["confidence_threshold"])

| # | GLiNER 단계 엔티티 · confidence |
|---|---|
| 1 | GCP에서(ORGANIZATION) 3/3 · 0.87 |
| 2 | – |
| 3 | – |
| 4 | 오라클(PERSON) 3/3 · 0.86<br>IP 기능을(OBJECT) 3/3 · 0.50 |

**해석**

- GLiNER가 메시지 4개에서 반환한 엔티티는 세 종류뿐이며, 세 개 모두 잘못된 결과이다. 결정적(deterministic)으로 3회 모두 같은 값을 반환한다.
- 한국어 조사(`에서`, `을`)가 엔티티 범위에 포함된다. GLiNER의 범위가 공백 단위 토큰 경계를 따르기 때문으로 보이며, [언어별 비교 실험](02_entity_extraction_by_language.ipynb)의 띄어쓰기 프로브로 확인한다.
- SDK의 CONFIDENCE 병합은 `정규화된 이름::타입`이 같을 때만 중복으로 판정한다. 이름이나 타입이 다른 GLiNER 결과는 LLM 결과와 겹치지 않고 그대로 추가된다.
- 오류의 confidence가 0.86~0.87로 높아 `confidence_threshold` 조정으로는 걸러지지 않는다. 0.5 초과로 설정하면 `IP 기능을`만 제거된다.

### 엔티티 해소 결과

빈 그래프에서 메시지 단위로 실행한 해소 결정을 집계한다. `created`는 기존 엔티티와 병합되지 않고 새 노드로 저장된다는 의미이다.

In [ ]:
decisions = Counter(
    (run["message_index"], d["name"], d["type"], d["action"], d["matched_entity_name"])
    for run in report["runs"] if run["mode"] == "hybrid"
    for d in run["empty_graph_per_message_resolution"]
)
show_table(
    ["#", "엔티티", "타입", "결정", "병합 대상", "횟수"],
    [[*key[:4], key[4] or "–", f"{count}/{report['repeats']}"]
     for key, count in sorted(decisions.items())],
)

**해석**

Hybrid의 모든 엔티티가 `created`로 결정되었다. 같은 메시지 안의 `GCP에서`와 `GCP`는 병합되지 않으며, `오라클`도 PERSON과 ORGANIZATION 두 노드로 각각 저장된다.
노트북 01에서 그래프에 `GCP에서`, `오라클(PERSON)` 노드가 따로 나타나는 현상과 일치한다.
단, 이 결과는 기존 노드가 없는 상태의 메시지 단위 해소이며, 이미 저장된 엔티티가 있는 DB에서의 결과와는 다를 수 있다.

### 관계 endpoint 선택

메시지 4에서 생성된 관계가 동명 노드 중 어느 쪽에 연결되는지 확인한다.

In [ ]:
show_table(
    ["모드", "반복", "관계", "출발점", "도착점"],
    [[run["mode"], run["repeat"], edge["relation_type"],
      f"{edge['source']['name']}({edge['source']['type']})" if edge["source"] else "–",
      f"{edge['target']['name']}({edge['target']['type']})" if edge["target"] else "–"]
     for run in report["runs"] if run["message_index"] == 4
     for edge in run["local_relation_endpoints"]],
)

| 모드 | 반복 | 관계 | 출발점 | 도착점 |
|---|---|---|---|---|
| llm_only | 3 | RELATED_TO | 오라클(ORGANIZATION) | 고정 IP 기능(OBJECT) |
| hybrid | 2 | RELATED_TO | **오라클(PERSON)** | 고정 IP 기능(OBJECT) |

**해석**

LLM은 메시지 4의 관계를 각 모드에서 3회 중 1회만 생성했다. Hybrid에서는 그 1회에서 관계의 출발점이 잘못된 `오라클(PERSON)` 노드로 연결되었다.
GLiNER의 오류는 고립된 노드로 남는 데 그치지 않고, LLM이 만든 올바른 관계를 잘못된 노드로 옮길 수 있다.

### 실행 시간

메시지·모드별 실행 시간(초)이다. 첫 Hybrid 실행에는 GLiNER 모델 로딩이 포함되므로 지연 시간 벤치마크로 해석하지 않는다.

In [ ]:
show_table(
    ["#", "LLM-only (s)", "Hybrid (s)"],
    [[index, *(", ".join(str(r["elapsed_seconds"]) for r in report["runs"]
                         if r["mode"] == mode and r["message_index"] == index)
               for mode in ("llm_only", "hybrid"))]
     for index in range(1, len(report["messages"]) + 1)],
)

모델 로딩이 포함된 첫 실행(8.46초)을 제외하면 두 모드 모두 약 1.0~2.6초 범위이다. LLM 호출이 시간 대부분을 차지하며, CPU에서 실행한 GLiNER 단계가 더하는 시간은 측정 오차 범위에 가깝다. GLiNER 단독 지연 시간은 [언어별 비교 실험](02_entity_extraction_by_language.ipynb)에서 측정한다.

## 결론과 한계

**결론**

1. 노트북 01의 `GCP에서`, `오라클(PERSON)`, `IP 기능을`은 GLiNER(`gliner2.5-base-v1`) 단계의 오류이며, CONFIDENCE 병합과 엔티티 해소를 거쳐 별도 노드로 저장된다.
2. 이 4개 메시지에서 Hybrid는 LLM-only보다 정답을 더 찾지 못했고, 오류 엔티티와 잘못된 관계 연결만 추가했다.
3. 오류의 confidence가 높아(0.86~0.87) threshold 조정은 해결책이 되지 않는다. 한국어 입력에는 LLM-only 구성, 다국어 체크포인트, 또는 조사 분리 전처리를 검토해야 한다.

**한계**

- 메시지 4개 × 3회의 진단용 표본이며, 정답 레이블이 없는 관찰 기록이다.
- 엔티티 해소는 빈 그래프에서 메시지 단위로 실행했으므로 DB 전체 흐름(end-to-end) 검증이 아니다.
- 관계 endpoint 추적은 저장 전 단계만 재현하며, 병합 이후의 ID 재매핑은 포함하지 않는다.
- 저장 필터 재현은 SDK 내부 메서드(`_apply_ontology`, `_local_endpoints`)에 의존한다.